In [1]:
Attach("IsogenyGraphBuilders.m");
Attach("MagmaAddOns.m");

Z:=Integers();

In [193]:
function Component0(v)
/*
    Compute the component of a directed graph from a vertex. 
*/
    function neighbor_step(V0,V1)
        W0:=&join[OutNeighbors(v) join InNeighbors(v) : v in V1] join V0 join V1;
        W1:={@ w : w in W0 | (not w in V1) and (not w in V0) @};
        return W0,W1;
    end function;
    
    V0:={@ v @};
    V1:={@ v @};
    repeat
        W0:=V0;
        W1:=V1;
        V0,V1:=neighbor_step(W0,W1);
    until #W0 eq #V0;

    return W0;
end function; 

function ConnectedComponents(G)
    comps:=[];
    all_verts:=Vertices(G);
    used_verts:={@ @};
    while #all_verts gt 0 do
        v:=all_verts[1];
        C:=Component(v);
        all_verts:={@ v : v in all_verts | not v in Vertices(C) @};
        used_verts:=used_verts join Vertices(C);
        Append(~comps,C);
    end while;
end function;


function GraphOverOrders0_l(R,l)

    GR:=GraphOverOrders0(R);
    ER:=Edges(GR);
    VR:=Vertices(GR);
    
    good_edges:=[];
    for e in ER do
        v0:=InitialVertex(e);
        v1:=TerminalVertex(e);
        O0:=ords[Index(VR,v0)];
        O1:=ords[Index(VR,v1)];
        if Index(O1,O0) eq l then
            Append(~good_edges,e);
        end if;
    end for;

    /*
    I couldn't get sub<GR|good_edges> or anything like it to work. 
    */
    
    bad_edges:={@ e : e in ER| not e in good_edges@};
    
    HR:=GR;
    for e in bad_edges do
        HR:= HR-e;
    end for;

    return HR;

end function;

In [194]:
P<x>:=PolynomialRing(Z);
//L:=1 - 2*x + 3*x^2 - 4*x^3 + 4*x^4; //boring
//L:=1 - 3*x + 5*x^2 - 9*x^3 + 9*x^4;
L:=1 + 2*x - x^2 - 5*x^3 - 8*x^4 - 15*x^5 - 9*x^6 + 54*x^7 + 81*x^8; //4.3.c_ab_af_ai growing example in paper
f:=WeilPolynomial(L);
//f:=x^4-2*x^2+121;
g,q,p:=Getgqp(f);
g,q,p;

K:=EtaleAlgebra(f);
F:=PrimitiveElement(K);
V:=q/F;
R:=Order([F,V]);
t0:=Cputime();
vert,edges:=IsogenyGraphBuilder(R,2);
t1:=Cputime(t0);
G:=ConstructStandardGrphMultDir(vert,edges);
t1;

4 3 3
2.160


In [195]:
V:=Vertices(G);
E:=Edges(G);

In [196]:
ords:={@ MultiplicatorRing(v[1]) : v in vert @};
#ords eq #OverOrders(R);
ords:=OverOrders(R);

true


In [197]:
//GR:=GraphOverOrders(R);

In [198]:
GR:=GraphOverOrders0(R);
ER:=Edges(GR);
VR:=Vertices(GR);

In [199]:
levels:=AssociativeArray();
order_index:=AssociativeArray();
over_orders:=OverOrders(R);

for i in [1..#vert] do 
    I:=vert[i][1];
    OI:=MultiplicatorRing(I);
    j:=Index(over_orders,OI);
    order_index[i]:=j;
    if j in Keys(levels) then
        Append(~levels[j],i);
    else 
        levels[j]:=[i];
    end if;
end for;

In [200]:
levels[1];
levels[2];
levels[3];
levels[4];
levels[5];
levels[6];
levels[7];
levels[8];

[ 1, 2, 3, 4, 5, 6 ]
[ 12 ]
[ 9 ]
[ 13, 14 ]
[ 11 ]
[ 10 ]
[ 8 ]
[ 7 ]


In [201]:
ascending:=[];
descending:=[];
horizontal:=[];

for e in E do
    v:=InitialVertex(e);
    w:=TerminalVertex(e);
    vR:=VR!order_index[Index(v)];
    wR:=VR!order_index[Index(w)];
    assert ( (vR eq wR) or (vR adj wR) or (wR adj vR) );
    if vR eq wR then 
        Append(~horizontal,e);
    elif vR adj wR then
        Append(~ascending,e);
    elif wR adj vR then
        Append(~descending,e);
    end if;
end for;

/*
//an element not in the graph;
//VR!1 adj VR!5;
//ER!e2;
*/

ascending;
descending;


[ < [1, 13], 13 >, < [2, 14], 16 >, < [3, 13], 14 >, < [4, 14], 17 >, < [5, 13], 15 >, < [6, 14], 18 >, < [7, 8], 3 >, < [9, 10], 10 >, < [12, 11], 11 > ]
[ < [8, 7], 4 >, < [10, 9], 2 >, < [11, 12], 12 >, < [13, 1], 1 >, < [13, 3], 7 >, < [13, 5], 8 >, < [14, 2], 6 >, < [14, 4], 5 >, < [14, 6], 9 > ]


In [202]:
r:=2;
f2:=WeilBaseChange(f,r);
g2,q2,p2:=Getgqp(f2);
assert q2 eq q^r;
g2,q2,p2;

4 9 3


In [203]:
K:=EtaleAlgebra(f2);
F:=PrimitiveElement(K);
V:=q2/F;
R:=Order([F,V]);
t0:=Cputime();
vert,edges:=IsogenyGraphBuilder(R,2);
t1:=Cputime(t0);
G2:=ConstructStandardGrphMultDir(vert,edges);
t1;

644.840


In [204]:
G:=G2;
E:=Edges(G);
V:=Vertices(G);

In [205]:
ords:=OverOrders(R);

In [207]:
HR:=GraphOverOrders0_l(R,2);
comps:=AllComponents0(HR);

frob_index:=Index(OverOrders(R),R);
max_index:=Index(OverOrders(R),MaximalOrder(K));

v_frob:=Vertices(HR)!frob_index;
v_max:=Vertices(HR)!max_index;

C_frob:=Component(v_frob);
C_max:=Component(v_max);

Edges(C_frob);
Edges(C_max);


In line 2, column 8:
>> comps:=AllComponents0(HR);
          ^
User error: Identifier 'AllComponents0' has not been declared or assigned
{@ [1, 6], [2, 4], [3, 2], [5, 3], [6, 5] @}
{@ @}


In [106]:
levels:=AssociativeArray();
order_index:=AssociativeArray();
over_orders:=OverOrders(R);

for i in [1..#vert] do 
    I:=vert[i][1];
    OI:=MultiplicatorRing(I);
    j:=Index(over_orders,OI);
    order_index[i]:=j;
    if j in Keys(levels) then
        Append(~levels[j],i);
    else 
        levels[j]:=[i];
    end if;
end for;

In [107]:
for i in [1..#over_orders] do
    i, #levels[i];
end for;

1 6
2 6
3 576
4 8
5 1
6 24
7 24
8 36
9 6
10 2
11 12
12 12
13 1
14 1
15 8
16 8
17 6
18 6
19 2
20 4
21 12
22 12
23 1
24 2
25 2
26 72
27 48
28 48
29 12
30 96
31 96
32 1
33 48
34 48
35 24
36 24
37 1
38 1
39 1
40 1
41 4
42 4
43 144
44 6
45 6
46 6
47 6
48 288


In [112]:
ascending:=[];
descending:=[];
horizontal:=[];
//neither:=[];

for e in E do
    v:=InitialVertex(e);
    w:=TerminalVertex(e);
    vR:=VR!order_index[Index(v)];
    wR:=VR!order_index[Index(w)];
    assert ( (vR eq wR) or (vR adj wR) or (wR adj vR) );
    if vR eq wR then 
        Append(~horizontal,e);
    elif vR adj wR then
        Append(~ascending,e);
    elif wR adj vR then
        Append(~descending,e);
    end if;
end for;

/*
//an element not in the graph;
//VR!1 adj VR!5;
//ER!e2;
*/

In [113]:
#ascending;
#descending;
#horizontal;
#neither;

1728
1728
0
0


In [115]:
#comps;

35


In [138]:
all_verts:=Vertices(G);
v:=all_verts[1];


In [139]:
W0:=Component0(v);

{ 783, 754, 987, 1, 408, 409, 759, 790, 994, 1750, 301, 766, 771, 975, 747, 778, 982 }


In [149]:
C:=Component(v);
VC:=Vertices(C);
Vertices(G)!VC[4];

409


In [152]:
[ #PicardGroup(T) : T in OverOrders(R)];
#OverOrders(R);

[ 6, 6, 576, 8, 1, 24, 24, 36, 6, 2, 12, 12, 1, 1, 8, 8, 6, 6, 2, 4, 12, 12, 1, 2, 2, 72, 48, 48, 12, 96, 96, 1, 48, 48, 24, 24, 1, 1, 1, 1, 4, 4, 144, 6, 6, 6, 6, 288 ]
48


In [157]:
GR:=GraphOverOrders0(R);
for e in Edges(GR) do
    printf "(%o,%o),\n",InitialVertex(e),TerminalVertex(e);
end for;

(1,9),
(1,32),
(2,10),
(2,37),
(2,38),
(3,30),
(3,31),
(3,34),
(3,48),
(4,20),
(5,32),
(6,11),
(6,21),
(6,24),
(7,12),
(7,21),
(7,25),
(8,2),
(8,17),
(8,18),
(8,29),
(9,23),
(10,39),
(10,40),
(11,13),
(11,17),
(11,46),
(12,14),
(12,18),
(12,46),
(13,5),
(13,37),
(14,5),
(14,38),
(15,4),
(15,41),
(16,4),
(16,42),
(17,1),
(17,37),
(17,44),
(18,1),
(18,38),
(18,45),
(19,5),
(20,19),
(21,19),
(21,46),
(22,24),
(22,25),
(22,47),
(24,13),
(24,19),
(25,14),
(25,19),
(26,8),
(26,11),
(26,12),
(26,47),
(27,6),
(27,35),
(27,41),
(28,7),
(28,35),
(28,42),
(29,10),
(29,44),
(29,45),
(30,15),
(30,27),
(30,33),
(31,16),
(31,28),
(31,33),
(32,23),
(33,4),
(33,35),
(34,15),
(34,16),
(34,36),
(35,20),
(35,21),
(36,22),
(36,41),
(36,42),
(37,32),
(37,39),
(38,32),
(38,40),
(39,23),
(40,23),
(41,20),
(41,24),
(42,20),
(42,25),
(43,6),
(43,7),
(43,22),
(43,26),
(44,9),
(44,39),
(45,9),
(45,40),
(46,1),
(46,5),
(47,2),
(47,13),
(47,14),
(48,27),
(48,28),
(48,36),
(48,43),


In [155]:
Edges(GR);

{@ [1, 9], [1, 32], [2, 10], [2, 37], [2, 38], [3, 30], [3, 31], [3, 34], [3, 48], [4, 20], [5, 32], [6, 11], [6, 21], [6, 24], [7, 12], [7, 21], [7, 25], [8, 2], [8, 17], [8, 18], [8, 29], [9, 23], [10, 39], [10, 40], [11, 13], [11, 17], [11, 46], [12, 14], [12, 18], [12, 46], [13, 5], [13, 37], [14, 5], [14, 38], [15, 4], [15, 41], [16, 4], [16, 42], [17, 1], [17, 37], [17, 44], [18, 1], [18, 38], [18, 45], [19, 5], [20, 19], [21, 19], [21, 46], [22, 24], [22, 25], [22, 47], [24, 13], [24, 19], [25, 14], [25, 19], [26, 8], [26, 11], [26, 12], [26, 47], [27, 6], [27, 35], [27, 41], [28, 7], [28, 35], [28, 42], [29, 10], [29, 44], [29, 45], [30, 15], [30, 27], [30, 33], [31, 16], [31, 28], [31, 33], [32, 23], [33, 4], [33, 35], [34, 15], [34, 16], [34, 36], [35, 20], [35, 21], [36, 22], [36, 41], [36, 42], [37, 32], [37, 39], [38, 32], [38, 40], [39, 23], [40, 23], [41, 20], [41, 24], [42, 20], [42, 25], [43, 6], [43, 7], [43, 22], [43, 26], [44, 9], [44, 39], [45, 9], [45, 40], [46, 1